# Performance analytics

**Prerequisites:** complete `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, `01_foundations/market_data_and_curves.ipynb`, and `01_foundations/math_toolkit.ipynb` in this curriculum (foundations track: core types, dates/calendars, market data/curves, and the math toolkit). This notebook assumes you are comfortable importing `finstack_quant` and working with simple Python lists.

**In this notebook:** return-based performance measurement — standalone metrics on `list[float]` inputs and the panel-level `Performance` engine (pandas-backed) for multi-asset summaries, correlations, rolling risk ratios, and drawdown episodes.

## Performance measurement in a nutshell

**Returns** are the basic input: from a price series you compute simple period returns, then compound them, measure dispersion (volatility), and compare reward to risk (Sharpe, Sortino, Calmar).

**Drawdowns** track peak-to-trough loss along a cumulative path; duration and depth matter as much as the headline max drawdown.

**Tail risk** (VaR and expected shortfall) summarizes how bad the left tail of the return distribution is — useful for risk limits and stress-style intuition.

Finstack Quant exposes fast Rust-backed implementations in `finstack_quant.analytics`: lightweight **standalone functions** for vectors, and a **`Performance`** class when you have a dated panel (multiple tickers, shared calendar).

### Standalone functions on return (and price) lists

Construct `Performance` from a price (or return) panel. Every metric — return/risk scalars, drawdowns, periodic returns, benchmark alpha/beta, basic factor models — is reachable as a method on the resulting instance.

Below: compounding, mean return, volatility (raw and annualized), classic ratios, drawdowns from returns, and a few distribution / streak metrics. Every result is printed so you can scan the magnitudes.

CAGR annualized from very short windows is unreliable

In [1]:
from datetime import date, timedelta

from finstack_quant.analytics import Performance

prices = [100.0, 105.0, 103.0, 108.0, 107.0, 112.0, 110.0, 115.0]
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(len(prices))]
perf = Performance.from_arrays(dates, [prices], ["SAMPLE"])

print("Active dates count:", len(perf.dates()))
print("CAGR:", round(perf.cagr()[0], 6))
print("Mean return (ann.):", round(perf.mean_return()[0], 6))
print("Volatility (per-period):", round(perf.volatility(annualize=False)[0], 6))
print("Volatility (ann.):", round(perf.volatility(annualize=True)[0], 6))
print("Sharpe:", round(perf.sharpe()[0], 6))
print("Sortino:", round(perf.sortino()[0], 6))
print("Max drawdown:", round(perf.max_drawdown()[0], 6))
print("Calmar:", round(perf.calmar()[0], 6))
print("Omega ratio:", round(perf.omega_ratio()[0], 6))
print("Gain-to-pain:", round(perf.gain_to_pain()[0], 6))
print("Downside deviation:", round(perf.downside_deviation()[0], 6))
print("Skewness:", round(perf.skewness()[0], 6))
print("Kurtosis:", round(perf.kurtosis()[0], 6))
print("Geometric mean:", round(perf.geometric_mean()[0], 6))

Active dates count: 7
CAGR: 1468.354538
Mean return (ann.): 5.204275
Volatility (per-period): 0.033882
Volatility (ann.): 0.537867
Sharpe: 9.67576
Sortino: 31.310607
Max drawdown: -0.019048
Calmar: 77088.613219
Omega ratio: 4.131512
Gain-to-pain: 3.131512
Downside deviation: 0.166214
Skewness: -0.398722
Kurtosis: -2.683661
Geometric mean: 0.020167


### Additional ratios (treynor, m-squared, sterling, pain, ulcer, recovery, skew/kurt combined)

A few more widely-used scalars not shown in the first pass.


In [2]:
print("Treynor:", round(perf.treynor()[0], 6))
print("M-squared:", round(perf.m_squared()[0], 6))
print("Sterling (n=3):", round(perf.sterling_ratio(n=3)[0], 6))
print("Pain ratio:", round(perf.pain_ratio()[0], 6))
print("Ulcer index:", round(perf.ulcer_index()[0], 6))
print("Recovery factor:", round(perf.recovery_factor()[0], 6))
sk, ku = perf.skew_kurt()
print("Skew/Kurt (tuple):", round(sk[0], 4), round(ku[0], 4))


Treynor: 5.204275
M-squared: 5.204275
Sterling (n=3): 95422.00834
Pain ratio: 222651.352794
Ulcer index: 0.010471
Recovery factor: 7.875
Skew/Kurt (tuple): -0.3987 -2.6837


### VaR, expected shortfall, and parametric variants

**Historical VaR** uses the empirical quantile of returns. In this library, `confidence` is the *VaR confidence level* in \((0, 1)\): for example `0.95` means **95% VaR**, i.e. the loss threshold at the **5th percentile** of the return distribution (see the Rust docstring for `value_at_risk`).

**Expected shortfall** (CVaR) averages returns worse than that threshold. **Parametric** and **Cornish–Fisher** variants adjust for Gaussian or skew/kurtosis structure.

All of the following print on the same `rets` sample as above.

In [3]:
print("Historical VaR (95% confidence):", round(perf.value_at_risk(0.95)[0], 6))
print("Expected shortfall (95%):", round(perf.expected_shortfall(0.95)[0], 6))
print("Parametric VaR (95%):", round(perf.parametric_var(0.95)[0], 6))
print("Cornish–Fisher VaR (95%):", round(perf.cornish_fisher_var(0.95)[0], 6))

Historical VaR (95% confidence): -0.01869
Expected shortfall (95%): -0.019048
Parametric VaR (95%): -0.03508
Cornish–Fisher VaR (95%): -0.040654


### `Performance`: dated panel analytics

`Performance` is constructed from a **pandas** `DataFrame` of prices (or via `Performance.from_arrays` with parallel lists). It aligns tickers on a calendar and returns **per-ticker vectors** for most metrics (`cagr()`, `sharpe()`, …).

Use `rolling_sharpe(ticker_idx, ...)`, `rolling_volatility(...)`, and `drawdown_details(ticker_idx, n=...)` when you need **time series** or **episode lists**. `period_stats(ticker_idx, agg_freq=...)` summarizes winning/losing streaks at a chosen aggregation.

In [4]:
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(252)]
spy_prices = [100.0]
for i in range(251):
    spy_prices.append(spy_prices[-1] * (1.0 + 0.0004 + (i % 17) * 1e-5))

perf = Performance.from_arrays(dates, [spy_prices], ["SPY"])

print("Tickers:", perf.ticker_names)
print("Freq:", perf.freq)
print("Dates count:", len(perf.dates()))

cagr_v = perf.cagr()
print("CAGR:", [round(x, 6) for x in cagr_v])
print("Mean return (ann.):", [round(x, 6) for x in perf.mean_return()])
print("Sharpe:", [round(x, 6) for x in perf.sharpe()])
print("Sortino:", [round(x, 6) for x in perf.sortino()])
print("Max drawdown:", [round(x, 6) for x in perf.max_drawdown()])
print("Max DD duration (days):", perf.max_drawdown_duration())
print("Volatility (ann.):", [round(x, 6) for x in perf.volatility()])
print("Calmar:", [round(x, 6) for x in perf.calmar()])

Tickers: ['SPY']
Freq: daily
Dates count: 251
CAGR: [0.191126]
Mean return (ann.): [0.120699]
Sharpe: [155.940601]
Sortino: [inf]
Max drawdown: [0.0]
Max DD duration (days): [0]
Volatility (ann.): [0.000774]
Calmar: [inf]


### Rolling metrics, drawdown paths, and period stats

`drawdown_series()` returns a drawdown path per ticker. `drawdown_details` lists the worst episodes with start, valley, and (if recovered) end dates.

Rolling helpers return objects with `.values` and `.dates()` for plotting or tables; here we print compact summaries.

In [5]:
dd_series = perf.drawdown_series()
print("First ticker drawdown series (first 5):", [round(x, 6) for x in dd_series[0][:5]])

episodes = perf.drawdown_details(0, n=3)
print("Top drawdown episodes (SPY):")
for i, ep in enumerate(episodes, start=1):
    end = ep.end
    print(
        f"  {i}) start={ep.start} valley={ep.valley} end={end!s} "
        f"days={ep.duration_days} max_dd={round(ep.max_drawdown, 6)}"
    )

rs = perf.rolling_sharpe(0, window=63, risk_free_rate=0.0)
rv = perf.rolling_volatility(0, window=63)
print("Rolling Sharpe: n=", len(rs.values), "first3=", [round(x, 6) for x in rs.values[:3]])
print("Rolling vol: n=", len(rv.values), "last3=", [round(x, 6) for x in rv.values[-3:]])

ps = perf.period_stats(0, agg_freq="monthly")
print(
    "Monthly period stats — win_rate:",
    round(ps.win_rate, 4),
    "best:",
    round(ps.best, 6),
    "worst:",
    round(ps.worst, 6),
    "consecutive_wins:",
    ps.consecutive_wins,
    "consecutive_losses:",
    ps.consecutive_losses,
)

First ticker drawdown series (first 5): [0.0, 0.0, 0.0, 0.0, 0.0]
Top drawdown episodes (SPY):
Rolling Sharpe: n= 189 first3= [np.float64(157.212368), np.float64(160.035287), np.float64(161.809907)]
Rolling vol: n= 189 last3= [np.float64(0.000777), np.float64(0.000762), np.float64(0.000751)]
Monthly period stats — win_rate: 1.0 best: 0.015048 worst: 0.003887 consecutive_wins: 9 consecutive_losses: 0


## Mini-example: three-ticker synthetic panel

We simulate **252 business days** of prices for **SPY**, **QQQ**, and **BND** with a simple Gaussian random walk: higher drift/vol for equities, muted dynamics for bonds. The panel comes from the example-only helper `_shared.random_walk_panel`, which is seeded so the notebook is reproducible.

We then rank key ratios per ticker, show the **correlation matrix**, summarize **rolling Sharpe** for SPY, and print **drawdown episodes** for each name.

In [6]:
import sys
sys.path.insert(0, "..")

from _shared import random_walk_panel

tickers = ["SPY", "QQQ", "BND"]

# Deterministic business-day random walk: one seeded stream, per-series drift/vol.
walk = random_walk_panel(
    252,
    3,
    seed=42,
    drift=[0.00035, 0.00055, 0.00008],
    vol=[0.008, 0.012, 0.0015],
    names=tickers,
)
panel = Performance.from_arrays(walk.dates, walk.prices, walk.names)

cagr_ = panel.cagr()
sharpe_ = panel.sharpe()
sortino_ = panel.sortino()
mdd_ = panel.max_drawdown()

print("Per-ticker headline metrics:")
for name, cg, sh, so, dd in zip(tickers, cagr_, sharpe_, sortino_, mdd_):
    print(
        f"  {name}: CAGR={round(cg, 6)} Sharpe={round(sh, 6)} Sortino={round(so, 6)} maxDD={round(dd, 6)}"
    )

corr = panel.correlation_matrix()
print("Correlation matrix (rows/cols =", tickers, "):")
for row in corr:
    print("  ", [round(x, 4) for x in row])

rs_spy = panel.rolling_sharpe(0, window=63)
print(
    "SPY rolling Sharpe: len=",
    len(rs_spy.values),
    "sample tail=",
    [round(x, 6) for x in rs_spy.values[-3:]],
)

for idx, name in enumerate(tickers):
    dds = panel.drawdown_details(idx, n=2)
    print(f"Drawdown details ({name}), top {len(dds)} episodes:")
    for j, ep in enumerate(dds, start=1):
        print(
            f"  {j}) max_dd={round(ep.max_drawdown, 6)} start={ep.start} valley={ep.valley} end={ep.end!s}"
        )


Per-ticker headline metrics:
  SPY: CAGR=0.329176 Sharpe=2.283121 Sortino=3.640353 maxDD=-0.047188
  QQQ: CAGR=0.078326 Sharpe=0.471581 Sortino=0.68924 maxDD=-0.147538
  BND: CAGR=-0.004867 Sharpe=-0.187369 Sortino=-0.271126 maxDD=-0.031884
Correlation matrix (rows/cols = ['SPY', 'QQQ', 'BND'] ):
   [1.0, 0.0818, -0.0477]
   [0.0818, 1.0, -0.0055]
   [-0.0477, -0.0055, 1.0]
SPY rolling Sharpe: len= 189 sample tail= [np.float64(1.925536), np.float64(1.647319), np.float64(1.624585)]
Drawdown details (SPY), top 2 episodes:
  1) max_dd=-0.047188 start=2024-11-20 valley=2024-12-06 end=None
  2) max_dd=-0.046757 start=2024-02-09 valley=2024-02-19 end=2024-03-25
Drawdown details (QQQ), top 2 episodes:
  1) max_dd=-0.147538 start=2024-09-18 valley=2024-11-29 end=None
  2) max_dd=-0.113402 start=2024-01-15 valley=2024-03-05 end=2024-05-24
Drawdown details (BND), top 2 episodes:
  1) max_dd=-0.031884 start=2024-05-20 valley=2024-10-21 end=None
  2) max_dd=-0.011249 start=2024-04-02 valley=2024-0

## Alternative constructors and DataFrame exports

`Performance` supports multiple construction paths:

- `Performance(prices_df)` — direct from a pandas DataFrame of prices (date index, one column per ticker).
- `Performance.from_returns(returns_df)` — **bring your own returns**: use this when your upstream system already produces a return stream (accounting/NAV feed, risk warehouse, vendor file) and prices are not the natural input.
- `from_arrays` / `from_returns_arrays` — from raw lists (shown above).

The cell below feeds `from_returns` with `prices_df.pct_change()` purely so the two constructor paths can be compared on the same data — pandas is standing in for "your returns source". If you want the library to do the differencing, `finstack_quant.features.transform_timeseries` computes returns (among many other panel transforms) in Rust; see `03_analytics/feature_transforms.ipynb`.

Export helpers turn the engine into tables for downstream use:

- `summary_to_dataframe()` — scalar metrics for all tickers as a DataFrame.
- `cumulative_returns_to_dataframe()` / `periodic_returns_to_dataframe(freq)` — time series and bucketed returns.


In [7]:
import pandas as pd

# Build a tiny price panel as a DataFrame (same shape as the 3-ticker example above)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(10)]
px = {
    "SPY": [100.0 + i*0.1 for i in range(10)],
    "BND": [100.0 - i*0.02 for i in range(10)],
}
prices_df = pd.DataFrame(px, index=dates)
prices_df.index.name = "date"

perf_df = Performance(prices_df, benchmark_ticker="SPY", freq="daily")
print("Direct DF ctor tickers:", perf_df.ticker_names)
print("Direct DF ctor CAGR:", [round(x, 6) for x in perf_df.cagr()])

# Side-by-side: from_arrays (earlier), direct DF ctor (above), and from_returns (here).
# pct_change() stands in for whatever produces returns in your pipeline; the point
# of the cell is the from_returns constructor, not the differencing.
ret_df = prices_df.pct_change().dropna()
perf_ret = Performance.from_returns(ret_df, benchmark_ticker="SPY", freq="daily")
print("from_returns tickers:", perf_ret.ticker_names)
print("from_returns Sharpe:", [round(x, 6) for x in perf_ret.sharpe()])

# DataFrame exports
print("\nSummary (head):")
print(perf_df.summary_to_dataframe().to_string())

print("\nCumulative returns (first 3 rows):")
print(perf_df.cumulative_returns_to_dataframe().head(3).to_string())

Direct DF ctor tickers: ['SPY', 'BND']
Direct DF ctor CAGR: [0.438522, -0.070507]
from_returns tickers: ['SPY', 'BND']
from_returns Sharpe: [5819.692446, -28959.558355]

Summary (head):
         cagr  mean_return  volatility        sharpe    sortino     calmar  max_drawdown  value_at_risk  expected_shortfall  tracking_error  information_ratio  skewness  kurtosis  geometric_mean  downside_deviation  omega_ratio  gain_to_pain  ulcer_index  pain_index  recovery_factor  tail_ratio  r_squared
SPY  0.438522     0.250998    0.000043   5819.692446        inf        inf        0.0000       0.000992            0.000992        0.000000           0.000000  0.007201 -1.199922        0.000996            0.000000          inf           inf     0.000000       0.000              inf    1.007197   1.000000
BND -0.070507    -0.050440    0.000002 -28959.558355 -15.874506 -39.170463       -0.0018      -0.000200           -0.000200        0.000041       -7283.349489 -0.001447 -1.199997       -0.000200      

## Windowing, resets, and structured result access

Use `reset_date_range` / `reset_bench_ticker` to narrow the analysis window or change the benchmark without rebuilding. Result objects carry rich accessors (e.g. `LookbackReturns.fytd`).


In [8]:
# recreate a slightly longer panel for reset demo
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(30)]
px = [100.0 + i*0.2 for i in range(30)]
p = Performance.from_arrays(dates, [px], ["TICK"])
print("full active dates:", len(p.active_dates()))
p.reset_date_range(dates[5], dates[-3])
print("after reset_date_range:", len(p.active_dates()))

# lookback with FYTD
lb = p.lookback_returns(dates[-1], fiscal_year_start_month=1)
print("lookback mtd/qtd/ytd sample:", round(lb.mtd[0],6), round(lb.qtd[0],6), round(lb.ytd[0],6))
print("fytd present:", lb.fytd is not None)

# one excess / outperformance series
ex = p.excess_returns([0.0]*len(p.dates()))
print("excess series len:", len(ex[0]))


full active dates: 29
after reset_date_range: 23
lookback mtd/qtd/ytd sample: 0.045635 0.045635 0.045635
fytd present: True
excess series len: 23


### Named lookback result

`lookback_returns` returns a `LookbackReturns` object, so code can access MTD/QTD/YTD/FYTD fields by name.

In [9]:
from finstack_quant.analytics import LookbackReturns

print("LookbackReturns type:", LookbackReturns.__name__)
print("lb is LookbackReturns:", isinstance(lb, LookbackReturns))
print("YTD sample:", round(lb.ytd[0], 6))

LookbackReturns type: LookbackReturns
lb is LookbackReturns: True
YTD sample: 0.045635


## Structured result types

`Performance`'s structured methods return typed value objects rather than loose tuples: `period_stats` → `PeriodStats`, `beta` → `BetaResult`, `greeks` → `GreeksResult`, `rolling_greeks` → `RollingGreeks`, `rolling_volatility`/`rolling_sharpe`/… → `DatedSeries`, `drawdown_details` → `DrawdownEpisode`, and `multi_factor_greeks` → `MultiFactorResult`.

In [10]:
from finstack_quant.analytics import (
    PeriodStats,
    BetaResult,
    GreeksResult,
    RollingGreeks,
    MultiFactorResult,
    DrawdownEpisode,
    DatedSeries,
)

# Four seeded series: three tickers plus one whose *returns* stand in for an
# explanatory factor fed to multi_factor_greeks.
#
# No benchmark_ticker is passed, so the benchmark defaults to the first ticker
# ("A") and beta[0]/alpha[0] are self-referential (1.0 / 0.0). This cell is about
# the RESULT TYPES, not the values; pass benchmark_ticker="BENCH" (or call
# reset_bench_ticker) when the numbers matter.
_panel = random_walk_panel(
    180,
    4,
    seed=1,
    drift=[0.0004, 0.0002, 0.0003, 0.0],
    vol=0.01,
    names=["A", "B", "BENCH", "FACTOR"],
)
sp = Performance.from_arrays(_panel.dates, _panel.prices[:3], _panel.names[:3])

ps = sp.period_stats(0)
betas = sp.beta()
gk = sp.greeks()
rg = sp.rolling_greeks(0, window=30)
rv = sp.rolling_volatility(0, window=30)
dd = sp.drawdown_details(0, n=3)
mf = sp.multi_factor_greeks(0, [_panel.returns[3]])

print("period_stats           ->", type(ps).__name__, "| best:", round(ps.best, 4))
print("beta[0]                ->", type(betas[0]).__name__, "| beta:", round(betas[0].beta, 4))
print("greeks[0]              ->", type(gk[0]).__name__, "| alpha:", round(gk[0].alpha, 6))
print("rolling_greeks         ->", type(rg).__name__, "| points:", len(rg.betas))
print("rolling_volatility     ->", type(rv).__name__, "| col:", rv.value_column)
print("drawdown_details[0]    ->", type(dd[0]).__name__, "| max_dd:", round(dd[0].max_drawdown, 4))
print("multi_factor_greeks    ->", type(mf).__name__, "| r2:", round(mf.r_squared, 4))

period_stats           -> PeriodStats | best: 0.1249
beta[0]                -> BetaResult | beta: 1.0
greeks[0]              -> GreeksResult | alpha: 0.0
rolling_greeks         -> RollingGreeks | points: 150
rolling_volatility     -> DatedSeries | col: volatility
drawdown_details[0]    -> DrawdownEpisode | max_dd: -0.1278
multi_factor_greeks    -> MultiFactorResult | r2: 0.0092


## Takeaways

- **Performance is the single entry point:** Construct it once from prices (or returns) and access scalars (`volatility`, `sortino`, `max_drawdown`), tail metrics (`value_at_risk`, `expected_shortfall`), and panel views (`correlation_matrix`) as methods on the same instance.
- **Confidence, not alpha:** pass `confidence=0.95` for 95% historical VaR in this API (the implementation targets the \((1 - \text{confidence})\) quantile).
- **Rolling and episode views:** `rolling_sharpe`, `rolling_volatility`, `rolling_greeks`, and `drawdown_details` provide time-series and episode-level forensics directly on the same `Performance` instance.
- **Reproducibility:** fix random seeds when you simulate; keep calendars consistent (`daily` frequency) when annualizing or comparing to 252-day conventions.

Next steps: combine these metrics with real curves and calendars from the foundations notebooks, or wire returns from your own portfolio accounting pipeline.